%md
# Mixed-effects models

Session-level averages lose the trial-by-trial structure of the data. Here each trial is a separate row, and the models account for the fact that trials are nested within users - everyone has their own baseline, and everyone has their own drift across the session.

Depression scores are z-standardized (PHQ-9 and BDI-II separately), so coefficients are in units of "one standard deviation of depression." Trial position is rescaled to 0–1 per session.

Each model tries a random intercept and random slope per user. If the slope variance collapses toward zero, the model is refit with random intercept only. The `random_structure` column records which one was used.

For each fit the summary reports the coefficient, raw p-value, Benjamini-Hochberg FDR-corrected p-value, a pseudo-Cohen's d (coefficient divided by the square root of between-person plus residual variance), and the ICC (fraction of variance that is between-person rather than trial-to-trial noise).

PHQ-9 and BDI-II are run independently and compared at the end.

## Depression × valence

This notebook tests the core Beck cognitive-model prediction: do depressed users allocate their attention to negative versus positive images differently than to neutral images?

Each trial is reshaped into long format with one row per valence. The model is:

`outcome ~ depression_score * valence`

with neutral as the reference category, giving two interaction terms: negative-vs-neutral and positive-vs-neutral.

Five outcomes are tested, covering different facets of attentional bias: total dwell time (maintenance), dwell time in the first 500 ms (early orientation), fixation proportion (allocation), revisit count (re-engagement), and time to first fixation (orientation latency).

In [0]:
%pip install statsmodels -q

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd

from src.evaluation.lmm_valence import melt_outcome, fit_all, apply_fdr, plot_valence_effects

## 1. Load data and reshape into long format

In [0]:
OUTCOMES = {
    "dwell_time_ms": ["dwell_time_ms_negative", "dwell_time_ms_positive", "dwell_time_ms_neutral"],
    "dwell_time_500ms": ["dwell_time_500ms_negative", "dwell_time_500ms_positive", "dwell_time_500ms_neutral"],
    "fixation_proportion": ["fixation_proportion_negative", "fixation_proportion_positive", "fixation_proportion_neutral"],
    "revisit_count": ["revisit_count_negative", "revisit_count_positive", "revisit_count_neutral"],
    "ttff_ms": ["ttff_ms_negative", "ttff_ms_positive", "ttff_ms_neutral"],
}

In [0]:
scene_metrics = spark.table("anima.scene_metrics")
df_forms = spark.table("anima.forms")

df_stimulus = scene_metrics.filter(F.col("scene_type") == "stimulus")

w = Window.partitionBy("session_id").orderBy("scene_index")
numbered = df_stimulus.withColumn("trial_num", F.row_number().over(w))

all_value_cols = [c for cols in OUTCOMES.values() for c in cols]
cols = ["session_id", "scene_index", "trial_num"] + all_value_cols

df_wide = numbered.select(*cols).join(
    df_forms.select("session_id", "uid", "phq9_score", "bdi_score"),
    on="session_id", how="inner",
).toPandas()

df_wide["trial_norm"] = df_wide.groupby("session_id")["trial_num"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0
)
df_wide["phq9_z"] = (df_wide["phq9_score"] - df_wide["phq9_score"].mean()) / df_wide["phq9_score"].std()
df_wide["bdi_z"] = (df_wide["bdi_score"] - df_wide["bdi_score"].mean()) / df_wide["bdi_score"].std()

df_wide[["session_id", "uid", "phq9_score", "bdi_score"]].nunique()

In [0]:
id_vars = ["session_id", "scene_index", "trial_num", "trial_norm", "uid", "phq9_score", "bdi_score", "phq9_z", "bdi_z"]

long_dfs = {name: melt_outcome(df_wide, cols, id_vars) for name, cols in OUTCOMES.items()}

pd.Series({name: len(d) for name, d in long_dfs.items()})

## 2. PHQ-9

### 2.1 Fit all models

In [0]:
phq_results, phq_summary = fit_all(long_dfs, "phq9_z")
phq_summary = apply_fdr(phq_summary)
phq_summary

### 2.2 FDR-significant interactions (α = 0.05)

In [0]:
phq_summary[phq_summary["neg_int_pval_fdr"] < 0.05][
    ["outcome", "neg_int_coef", "neg_int_pval_fdr", "neg_int_d"]
]

In [0]:
phq_summary[phq_summary["pos_int_pval_fdr"] < 0.05][
    ["outcome", "pos_int_coef", "pos_int_pval_fdr", "pos_int_d"]
]

### 2.3 Dwell time by valence and PHQ-9 group

In [0]:
df_plot = long_dfs["dwell_time_ms"].rename(columns={"y": "dwell_time"})
plot_valence_effects(df_plot, "phq9_score", "PHQ-9")

## 3. BDI-II

### 3.1 Fit all models

In [0]:
bdi_results, bdi_summary = fit_all(long_dfs, "bdi_z")
bdi_summary = apply_fdr(bdi_summary)
bdi_summary

### 3.2 FDR-significant interactions (α = 0.05)

In [0]:
bdi_summary[bdi_summary["neg_int_pval_fdr"] < 0.05][
    ["outcome", "neg_int_coef", "neg_int_pval_fdr", "neg_int_d"]
]

In [0]:
bdi_summary[bdi_summary["pos_int_pval_fdr"] < 0.05][
    ["outcome", "pos_int_coef", "pos_int_pval_fdr", "pos_int_d"]
]

### 3.3 Dwell time by valence and BDI-II group

In [0]:
plot_valence_effects(df_plot, "bdi_score", "BDI-II")

## 4. Detailed fit for the headline outcome

In [0]:
print(phq_results["dwell_time_ms"].summary())
print()
print(bdi_results["dwell_time_ms"].summary())

## 5. PHQ-9 vs BDI-II comparison

In [0]:
comparison = phq_summary[["outcome", "neg_int_coef", "neg_int_pval_fdr", "pos_int_coef", "pos_int_pval_fdr", "icc"]].merge(bdi_summary[["outcome", "neg_int_coef", "neg_int_pval_fdr", "pos_int_coef", "pos_int_pval_fdr"]], on="outcome", suffixes=("_phq", "_bdi"),
)

comparison[["outcome", "neg_int_coef_phq", "neg_int_pval_fdr_phq", "neg_int_coef_bdi", "neg_int_pval_fdr_bdi"]]

In [0]:
comparison[["outcome", "pos_int_coef_phq", "pos_int_pval_fdr_phq", "pos_int_coef_bdi", "pos_int_pval_fdr_bdi"]]

In [0]:
neg_both = ((comparison["neg_int_pval_fdr_phq"] < 0.05) & (comparison["neg_int_pval_fdr_bdi"] < 0.05)).sum()
pos_both = ((comparison["pos_int_pval_fdr_phq"] < 0.05) & (comparison["pos_int_pval_fdr_bdi"] < 0.05)).sum()

pd.Series({
    "negative × score convergent": neg_both,
    "positive × score convergent": pos_both,
    "n_outcomes": len(comparison),
})

## 6. Reliability (ICC)

In [0]:
phq_summary[["outcome", "icc"]].sort_values("icc", ascending=False)

In [0]:
phq_summary["icc"].agg(["mean", "median"]).round(3)